# InSAR vs SNOTEL SWE

User-friendly workflow for comparing MintPy-derived InSAR SWE changes against cached SNOTEL station data from a previous notebook run.

## Quick start

1. Run the SNOTEL notebook first and save its pickle output.
2. Update the four file paths in the configuration cell.
3. Run all cells from top to bottom.
4. Review the station table.
5. Inspect the delta-SWE and cumulative-SWE comparison plots for all stations.

## What this notebook assumes

- You already created a SNOTEL pickle with `site_loc` geometry per station.
- You already produced a MintPy time-series file.
- You have an incidence-angle file that can be sampled at station locations.
- MintPy point extraction via `read_timeseries_lalo()` is valid for your time-series and lookup configuration.


In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from mintpy.utils import utils as ut

from snowsar.utils import compare_station_windows, load_pickle, save_pickle, snotel_site_table_from_results, write_mintpy_array_as_geotiff


## 1. Configuration

Edit this cell only for a normal run.


In [ ]:
# Pickle created by the SNOTEL notebook.
SNOTEL_PICKLE = Path("/path/to/cache/snotel_data_mintpy.pkl").expanduser()

# MintPy time-series file.
TIMESERIES_FILE = Path("/path/to/MintPy/timeseries/file").expanduser()

# Lookup or geometry file used by MintPy for metadata and optional plotting context.
LOOKUP_FILE = Path("/path/to/MintPy/geometry/file").expanduser()

# Incidence-angle HDF5 geometry file.
INCIDENCE_FILE = Path("/path/to/MintPy/geometry/file").expanduser()

# Coherence stack used to align MintPy neighbor intervals with ifgramStack pairs.
COHERENCE_FILE = Path("/path/to/MintPy/ifgramStack/file").expanduser()


## 2. Advanced Configuration

Leave these defaults alone unless you are adjusting extraction behavior.


In [ ]:
WIN_SIZE = 10
INCIDENCE_DATASET = "incidenceAngle"
COHERENCE_DATASET = "coherence"
COHERENCE_DATE_DATASET = "date"
COHERENCE_THRESHOLD = 0.35
APPLY_BIAS_CORRECTION = True
BIAS_STATION_NAMES: list[str] = []
SAVE_PLOTS = True
PLOT_OUTPUT_DIR = Path("figures/insar_swe_vs_snotel")
SAVE_SCENE_GEOTIFFS = True
SCENE_GEOTIFF_DIR = Path("outputs/scene_delta_swe_geotiffs")
SAVE_RESULTS = True
OUTPUT_FILE = Path("cache/insar_swe_vs_snotel.pkl")


## 3. Helper Functions


In [ ]:
def require(condition: bool, message: str, errors: list[str]) -> None:
    if not condition:
        errors.append(message)


def plot_station_filename(prefix: str, site_name: str) -> str:
    safe_name = "".join(ch if ch.isalnum() or ch in (" ", "-", "_") else "_" for ch in site_name).strip()
    safe_name = safe_name.replace("/", "_")
    return f"{prefix}_{safe_name}.png"


## 4. Preflight Check

This cell checks file paths and summarizes the MintPy file metadata before comparison starts.


In [ ]:
errors: list[str] = []

require(str(SNOTEL_PICKLE) != "/path/to/cache/snotel_data_mintpy.pkl", "Update SNOTEL_PICKLE in the configuration cell.", errors)
require(str(TIMESERIES_FILE) != "/path/to/MintPy/timeseries/file", "Update TIMESERIES_FILE in the configuration cell.", errors)
require(str(LOOKUP_FILE) != "/path/to/MintPy/geometry/file", "Update LOOKUP_FILE in the configuration cell.", errors)
require(str(INCIDENCE_FILE) != "/path/to/MintPy/geometry/file", "Update INCIDENCE_FILE in the configuration cell.", errors)
require(SNOTEL_PICKLE.exists(), f"SNOTEL pickle not found: {SNOTEL_PICKLE}", errors)
require(TIMESERIES_FILE.exists(), f"MintPy time-series file not found: {TIMESERIES_FILE}", errors)
require(LOOKUP_FILE.exists(), f"Lookup file not found: {LOOKUP_FILE}", errors)
require(INCIDENCE_FILE.exists(), f"Incidence file not found: {INCIDENCE_FILE}", errors)
require(COHERENCE_FILE.exists(), f"Coherence file not found: {COHERENCE_FILE}", errors)
require(WIN_SIZE >= 1, "WIN_SIZE must be at least 1.", errors)
require(0.0 <= COHERENCE_THRESHOLD <= 1.0, "COHERENCE_THRESHOLD must be between 0 and 1.", errors)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
if SAVE_PLOTS:
    PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if SAVE_SCENE_GEOTIFFS:
    SCENE_GEOTIFF_DIR.mkdir(parents=True, exist_ok=True)

if errors:
    raise RuntimeError("Preflight failed:\n- " + "\n- ".join(errors))

attrs = ut.readfile.read_attribute(str(TIMESERIES_FILE))
display(
    pd.DataFrame(
        [
            {
                "FILE_TYPE": attrs.get("FILE_TYPE"),
                "EPSG": attrs.get("EPSG"),
                "X_UNIT": attrs.get("X_UNIT"),
                "Y_UNIT": attrs.get("Y_UNIT"),
                "X_FIRST": attrs.get("X_FIRST"),
                "Y_FIRST": attrs.get("Y_FIRST"),
                "X_STEP": attrs.get("X_STEP"),
                "Y_STEP": attrs.get("Y_STEP"),
                "COHERENCE_THRESHOLD": COHERENCE_THRESHOLD,
                "APPLY_BIAS_CORRECTION": APPLY_BIAS_CORRECTION,
            }
        ]
    )
)

print("Preflight passed. Continue to load station data.")


## 5. Load Cached SNOTEL Data


In [ ]:
swe_data = load_pickle(SNOTEL_PICKLE)
site_table = snotel_site_table_from_results(swe_data)

if site_table.empty:
    raise RuntimeError(
        "No station coordinates were found in the cached SNOTEL pickle. "
        "Confirm the pickle came from the SNOTEL notebook and includes site_loc geometry."
    )

print(f"Stations loaded: {len(site_table)}")
display(site_table)


## 6. Run the InSAR vs SNOTEL Comparison


In [ ]:
sites_to_compare = site_table.copy()

all_results, failures_df = compare_station_windows(
    sites_to_compare,
    swe_data,
    timeseries_file=TIMESERIES_FILE,
    coherence_file=COHERENCE_FILE,
    incidence_file=INCIDENCE_FILE,
    incidence_dataset=INCIDENCE_DATASET,
    coherence_dataset=COHERENCE_DATASET,
    coherence_date_dataset=COHERENCE_DATE_DATASET,
    coherence_threshold=COHERENCE_THRESHOLD,
    apply_bias_correction=APPLY_BIAS_CORRECTION,
    bias_station_names=BIAS_STATION_NAMES,
    win_size=WIN_SIZE,
)

if not all_results:
    raise RuntimeError("No station comparisons succeeded. Inspect the logged failures for details.")

print(f"Successful stations: {len(all_results)}")
print(f"Failed stations: {len(failures_df)}")


## 7. Plot Whole-Scene Delta SWE


In [ ]:
with h5py.File(TIMESERIES_FILE, "r") as ts_h5, h5py.File(COHERENCE_FILE, "r") as coh_h5, h5py.File(INCIDENCE_FILE, "r") as inc_h5:
    ts_dates = [value.decode("utf-8") if isinstance(value, bytes) else str(value) for value in ts_h5["date"][...]]
    pair_index = {
        tuple(value.decode("utf-8") if isinstance(value, bytes) else str(value) for value in row): idx
        for idx, row in enumerate(coh_h5[COHERENCE_DATE_DATASET][...])
    }
    incidence_deg = np.asarray(inc_h5[INCIDENCE_DATASET][...], dtype=float)
    incidence_rad = np.deg2rad(incidence_deg)
    factor_grid = (-0.6784 * incidence_rad**2) + (0.2899 * incidence_rad) - 0.8473

    for i_day in range(len(ts_dates) - 1):
        date1 = ts_dates[i_day]
        date2 = ts_dates[i_day + 1]
        pair = (date1, date2)
        if pair not in pair_index:
            print(f"[skip] missing coherence pair for {date1} -> {date2}")
            continue

        displacement_delta = -(
            np.asarray(ts_h5["timeseries"][i_day + 1], dtype=float)
            - np.asarray(ts_h5["timeseries"][i_day], dtype=float)
        )
        coherence_map = np.asarray(coh_h5[COHERENCE_DATASET][pair_index[pair]], dtype=float)
        swe_map = displacement_delta / -factor_grid * 100.0
        swe_map = np.where(coherence_map == 0, np.nan, swe_map)
        finite = np.isfinite(swe_map)
        if not finite.any():
            print(f"[skip] no valid scene pixels for {date1} -> {date2}")
            continue

        vmax = float(np.nanpercentile(np.abs(swe_map[finite]), 98))
        vmax = max(vmax, 0.1)

        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(swe_map, cmap="RdBu", vmin=-vmax, vmax=vmax)
        ax.set_title(f"Delta SWE\n{date1} - {date2}")
        ax.set_xlabel("Column")
        ax.set_ylabel("Row")
        cbar = fig.colorbar(im, ax=ax)
        cbar.set_label("Delta SWE (cm)")
        fig.tight_layout()
        if SAVE_SCENE_GEOTIFFS:
            write_mintpy_array_as_geotiff(
                swe_map.astype(np.float32),
                TIMESERIES_FILE,
                SCENE_GEOTIFF_DIR / f"delta_swe_scene_{date1}_{date2}.tif",
                description=f"Delta SWE {date1} to {date2}",
            )
        if SAVE_PLOTS:
            fig.savefig(PLOT_OUTPUT_DIR / plot_station_filename("delta_swe", f"{date1}-{date2}"), dpi=200, bbox_inches="tight")
        plt.show()


## 8. Plot Delta SWE by Station


In [ ]:
for site_name, result in all_results.items():
    aligned = result["aligned"]
    metrics_row = result["metrics"].iloc[0]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(aligned["date"], aligned["insar_delta_swe_cm"], marker="^", label="InSAR")
    ax.plot(aligned["date"], aligned["snotel_delta_swe_cm"], marker="o", label="SNOTEL")
    ax.tick_params(which="both", direction="in", bottom=True, top=True, left=True, right=True)
    ax.set_xlabel("Date")
    ax.set_ylabel("Delta SWE (cm)")
    ax.legend()
    ax.set_title(site_name)
    ax.text(
        0.02,
        0.98,
        f"RMSE = {metrics_row['rmse_cm']:.2f} cm\n$R^2$ = {metrics_row['r2']:.2f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.75),
    )
    fig.autofmt_xdate()
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOT_OUTPUT_DIR / plot_station_filename("delta_swe", site_name), dpi=200, bbox_inches="tight")
    plt.show()


## 9. Plot Cumulative SWE


In [ ]:
for site_name, result in all_results.items():
    aligned = result["aligned"]
    y_true = aligned["snotel_cumulative_swe_cm"].to_numpy(dtype=float)
    y_pred = aligned["insar_cumulative_swe_cm"].to_numpy(dtype=float)
    resid = y_pred - y_true
    cumulative_rmse = np.sqrt(np.mean(resid**2)) if len(aligned) else np.nan
    denom = np.sum((y_true - y_true.mean()) ** 2)
    cumulative_r2 = 1.0 - (np.sum(resid**2) / denom) if len(aligned) and denom > 0 else np.nan

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(aligned["date"], aligned["insar_cumulative_swe_cm"], marker="^", label="InSAR")
    ax.plot(aligned["date"], aligned["snotel_cumulative_swe_cm"], marker="o", label="SNOTEL")
    ax.tick_params(which="both", direction="in", bottom=True, top=True, left=True, right=True)
    ax.set_xlabel("Date")
    ax.set_ylabel("Cumulative SWE (cm)")
    ax.legend()
    ax.set_title(site_name)
    ax.text(
        0.02,
        0.98,
        f"RMSE = {cumulative_rmse:.2f} cm\n$R^2$ = {cumulative_r2:.2f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.75),
    )
    fig.autofmt_xdate()
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOT_OUTPUT_DIR / plot_station_filename("total_swe", site_name), dpi=200, bbox_inches="tight")
    plt.show()


## 10. Summary Figure Across Stations and Interferograms


In [ ]:
rows = []
for site_name, result in all_results.items():
    aligned = result["aligned"].copy()
    aligned["site_name"] = site_name
    rows.append(
        aligned[
            [
                "site_name",
                "date_start",
                "date_end",
                "snotel_delta_swe_cm",
                "insar_delta_swe_cm",
            ]
        ]
    )

summary_df = pd.concat(rows, ignore_index=True)
summary_df = summary_df.dropna(subset=["snotel_delta_swe_cm", "insar_delta_swe_cm"]).copy()
if summary_df.empty:
    raise RuntimeError("No valid paired InSAR/SNOTEL delta SWE samples available for the summary figure.")

overall_true = summary_df["snotel_delta_swe_cm"].to_numpy(dtype=float)
overall_pred = summary_df["insar_delta_swe_cm"].to_numpy(dtype=float)
overall_rmse = np.sqrt(np.mean((overall_pred - overall_true) ** 2))
overall_corr = np.corrcoef(overall_true, overall_pred)[0, 1] if len(summary_df) > 1 else np.nan

def corr_and_mae(group: pd.DataFrame) -> pd.Series:
    values = group[["snotel_delta_swe_cm", "insar_delta_swe_cm"]].dropna()
    if values.empty:
        return pd.Series({"corr": np.nan, "mae_cm": np.nan, "pairs": 0})
    y_true = values["snotel_delta_swe_cm"].to_numpy(dtype=float)
    y_pred = values["insar_delta_swe_cm"].to_numpy(dtype=float)
    corr = np.corrcoef(y_true, y_pred)[0, 1] if len(values) > 1 else np.nan
    mae_cm = np.mean(np.abs(y_pred - y_true))
    return pd.Series({"corr": corr, "mae_cm": mae_cm, "pairs": len(values)})

station_stats = summary_df.groupby("site_name", sort=False).apply(corr_and_mae).reset_index()
station_stats["station_number"] = np.arange(1, len(station_stats) + 1)

interval_stats = (
    summary_df.groupby("date_start", sort=True).apply(corr_and_mae).reset_index().sort_values("date_start")
)
interval_stats["interval_number"] = np.arange(len(interval_stats))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

ax = axes[0]
ax.plot(summary_df["snotel_delta_swe_cm"], summary_df["insar_delta_swe_cm"], "*", markersize=6)
ax.grid(True, alpha=0.25)
ax.set_xlabel("Station Δ SWE (cm)")
ax.set_ylabel("Sentinel-1 Δ SWE (cm)")
ax.set_title("All Valid Station-Interval Pairs")
ax.text(
    0.04,
    0.90,
    f"Correlation: {overall_corr:.2f}\nRMSE: {overall_rmse:.2f} cm",
    transform=ax.transAxes,
    ha="left",
    va="top",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.75),
)

ax = axes[1]
ax2 = ax.twinx()
ax.plot(station_stats["station_number"], station_stats["corr"], "o", mfc="none", mec="tab:blue", mew=1.5)
ax2.plot(station_stats["station_number"], station_stats["mae_cm"], "o", mfc="none", mec="tab:orange", mew=1.5)
ax.grid(True, alpha=0.25)
ax.set_xlabel("Station #")
ax.set_ylabel("ρᵢ", color="tab:blue")
ax2.set_ylabel("<Δ SWE Err> (cm)", color="tab:orange")
ax.tick_params(axis="y", colors="tab:blue")
ax2.tick_params(axis="y", colors="tab:orange")
ax.set_title("Per-Station Correlation and Error")
ax.set_xlim(0.5, len(station_stats) + 0.5)
ax.set_ylim(-1.0, 1.0)
ax2.set_ylim(bottom=0.0)

ax = axes[2]
ax2 = ax.twinx()
ax.plot(interval_stats["interval_number"], interval_stats["corr"], "o", mfc="none", mec="tab:blue", mew=1.5)
ax2.plot(interval_stats["interval_number"], interval_stats["mae_cm"], "o", mfc="none", mec="tab:orange", mew=1.5)
ax.grid(True, alpha=0.25)
ax.set_xlabel("Observation Date")
ax.set_ylabel("ρᵢ", color="tab:blue")
ax2.set_ylabel("<Δ SWE Err> (cm)", color="tab:orange")
ax.tick_params(axis="y", colors="tab:blue")
ax2.tick_params(axis="y", colors="tab:orange")
ax.set_title("Per-Interferogram Correlation and Error")
ax.set_ylim(-1.0, 1.0)
ax2.set_ylim(bottom=0.0)
tick_step = max(1, len(interval_stats) // 6)
tick_positions = interval_stats["interval_number"].iloc[::tick_step].tolist()
if tick_positions[-1] != interval_stats["interval_number"].iloc[-1]:
    tick_positions.append(int(interval_stats["interval_number"].iloc[-1]))
tick_labels = [interval_stats.loc[interval_stats["interval_number"] == pos, "date_start"].iloc[0].strftime("%m/%d") for pos in tick_positions]
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOT_OUTPUT_DIR / "summary_all_stations.png", dpi=200, bbox_inches="tight")
plt.show()


## 11. Save Comparison Result


In [ ]:
if SAVE_RESULTS:
    payload = {
        "all_results": all_results,
        "failures": failures_df,
        "coherence_threshold": COHERENCE_THRESHOLD,
        "apply_bias_correction": APPLY_BIAS_CORRECTION,
        "bias_station_names": BIAS_STATION_NAMES,
        "timeseries_file": str(TIMESERIES_FILE),
        "coherence_file": str(COHERENCE_FILE),
    }
    save_pickle(payload, OUTPUT_FILE)
    print(f"Saved comparison result: {OUTPUT_FILE}")
else:
    print("SAVE_RESULTS is False. Skipping result export.")


## What Next

- Change `COHERENCE_THRESHOLD` or `BIAS_STATION_NAMES` to tune the MATLAB-style comparison workflow.
- If many intervals are skipped, inspect `ifgramStack.h5` date pairs and coherence coverage for the missing neighbor intervals.
- If projected station coordinates look suspicious, regenerate the SNOTEL cache from the updated MintPy/SNOTEL notebook before interpreting the SWE comparison.
